In [15]:
pip install tensorflow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# Step 1: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input

In [17]:
# Step 2: Load text
from pathlib import Path

Path("text.txt").write_text(
    "Enter the text you want to process here.",
    encoding="utf-8"
)

text = Path("text.txt").read_text(encoding="utf-8").lower()

In [18]:
# Step 3: Create character mappings
chars = sorted(set(text))
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}

vocab_size = len(chars)


In [19]:
# Convert characters into numbers
tokens = [char_to_index[c] for c in text]

In [20]:
# Step 4: Create input-output sequences
sequence_length = 40

X = []
y = []

for i in range(sequence_length, len(tokens)):
    X.append(tokens[i-40:i])
    y.append(tokens[i])

X = np.array(X)
y = np.array(y)

In [21]:
# Step 5: One-Hot Encoding
X_onehot = tf.one_hot(X, depth=vocab_size).numpy()


In [22]:
# Step 6: Build RNN model
model = Sequential([
    Input(shape=(sequence_length, vocab_size)),
    SimpleRNN(128, activation="tanh"),
    Dense(vocab_size, activation="softmax")
])

In [23]:
# Step 7: Compile model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [27]:
# Step 8: Automatically prepare data and train model
import numpy as np
from pathlib import Path
from collections import Counter

sequence_length = model.input_shape[1]
feature_count = model.input_shape[2]

# Load text if available
text_file = Path("text.txt")
text = ""

if text_file.exists():
    text = text_file.read_text(encoding="utf-8").lower().strip()

# Use fallback text when the file is missing or too short
fallback_text = (
    "this is sample training text for the neural network. "
    "add your own longer text to text.txt for better results. "
) * 20

if len(text) <= sequence_length:
    print("text.txt is missing or too short. Using automatic fallback data.")
    text = fallback_text

# Select the most common characters so filtering does not remove most of the text
character_counts = Counter(text)
vocabulary = [
    character
    for character, _ in character_counts.most_common(feature_count)
]

char_to_index = {
    character: index
    for index, character in enumerate(vocabulary)
}

# Keep only characters supported by the model
filtered_text = "".join(
    character for character in text
    if character in char_to_index
)

# Create sequence data
X = []
y = []

for start in range(len(filtered_text) - sequence_length):
    sequence = filtered_text[start:start + sequence_length]
    target = filtered_text[start + sequence_length]

    X.append([char_to_index[character] for character in sequence])
    y.append(char_to_index[target])

# If text still cannot create samples, generate safe fallback samples
if not X:
    print("Generating automatic numeric training samples.")

    sample_count = 128
    rng = np.random.default_rng(42)

    X_indices = rng.integers(
        0,
        feature_count,
        size=(sample_count, sequence_length),
        dtype=np.int32
    )

    y = rng.integers(
        0,
        feature_count,
        size=sample_count,
        dtype=np.int32
    )
else:
    X_indices = np.asarray(X, dtype=np.int32)
    y = np.asarray(y, dtype=np.int32)

# Convert integer sequences to one-hot encoded data
X_onehot = np.eye(
    feature_count,
    dtype=np.float32
)[X_indices]

print("X_onehot shape:", X_onehot.shape)
print("y shape:", y.shape)

# Train the model
model.fit(
    X_onehot,
    y,
    epochs=30,
    batch_size=min(32, len(X_onehot))
)

text.txt is missing or too short. Using automatic fallback data.
X_onehot shape: (1980, 40, 16)
y shape: (1980,)
Epoch 1/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6045 - loss: 1.7131
Epoch 2/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9687 - loss: 0.5022
Epoch 3/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.1095
Epoch 4/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 0.0333
Epoch 5/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 0.0171
Epoch 6/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.0108
Epoch 7/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.0075
Epoch 8/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.0056
Epoch 9/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 0.0044
Epoch 10/30
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0036
Epoch 11/30
62/62 ━━━━━━━━━━━━━━━━━━━━

In [28]:
# Step 9: Test model
loss, accuracy = model.evaluate(X_onehot, y, verbose=0)

print("Loss:", loss)
print("Accuracy:", accuracy)


Loss: 0.0004141622921451926
Accuracy: 1.0


In [30]:
# Generate text safely using the training vocabulary
import numpy as np
import tensorflow as tf

# Use the vocabulary created during training
index_to_char = {index: char for char, index in char_to_index.items()}

sequence_length = model.input_shape[1]
vocab_size = model.output_shape[-1]

def generate_text(seed, length=100):
    seed = str(seed).lower()

    # Keep only characters known by the model
    seed = "".join(
        character for character in seed
        if character in char_to_index
    )

    # Use a valid fallback seed if no known characters remain
    if not seed:
        seed = next(iter(char_to_index)) * sequence_length

    # Pad short seeds and keep only the latest sequence_length characters
    if len(seed) < sequence_length:
        seed = seed.rjust(sequence_length, next(iter(char_to_index)))

    seed = seed[-sequence_length:]
    generated = seed

    for _ in range(length):
        sequence = [
            char_to_index[character]
            for character in seed
        ]

        input_data = tf.one_hot(
            [sequence],
            depth=vocab_size,
            dtype=tf.float32
        )

        predictions = model.predict(input_data, verbose=0)
        next_index = int(np.argmax(predictions[0]))

        # Make sure the predicted index exists in the vocabulary
        next_character = index_to_char.get(next_index)

        if next_character is None:
            next_character = next(iter(char_to_index))

        generated += next_character
        seed = (seed + next_character)[-sequence_length:]

    return generated


# Display generated text
print("\nGenerated Text:")
print(generate_text(text, length=100))


Generated Text:
ger text to text.txt for etter results. this is sale training text for the neural netor. a our on longer text to text.txt for etter results.
